# Malladi & Jaiswal et al., Science Immunology, 2025 (transcriptomics)

Author: Julian Q. Zhou

Docker container: `julianqz/wu_cimm:ref_0.1.1_lsf` (Python 3.8.8)

The code below was run on a HPC environment that did not support running Jupyter Lab via a browser.

As such, key console outputs were pasted as comments. Visualizations were outputted as pdfs or pngs.

A tsv file was exported at the end for overlaying S-binding specificity on the UMAPs in the R script.

## Load packages & config

In [ ]:
from pathlib import Path
import os
import copy
import re
import math
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import read_h5ad
from anndata import AnnData
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# scanpy settings

# verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.verbosity = 3             
sc.logging.print_header()

In [ ]:
# scanpy==1.7.2 anndata==0.7.6 umap==0.5.1 numpy==1.20.2 scipy==1.6.2 pandas==1.2.3 scikit-learn==0.24.1 statsmodels==0.12.2 python-igraph==0.9.4

In [ ]:
sc.settings.set_figure_params(dpi=120, dpi_save=150, vector_friendly=False, format="pdf", 
                              transparent=False, facecolor="w", color_map="viridis_r")

In [ ]:
sc.settings.figdir = "."

In [ ]:
# resolution for clustering all cells
res_1 = 0.18
anno_col_1 = f"anno_leiden_{res_1:.2f}"

In [ ]:
# resolution for clustering B cells
res_2 = 0.25
anno_col_2 = f"anno_leiden_{res_2:.2f}"

In [ ]:
# number of principal components to use
N_PC = 20

In [ ]:
# type of embedding
eb = "umap"

In [ ]:
# filenames
fn_1 = "WU382_malladi_et_al_sci_imm_2025_gex_all_cells.h5ad"
fn_2 = "WU382_malladi_et_al_sci_imm_2025_gex_b_cells.h5ad"
fn_save = f"WU382_malladi_et_al_sci_imm_2025_gex_b_cell_{eb}.tsv.gz"

## All cells

### AnnData containing clustering results of all cells

In [ ]:
adata_1 = read_h5ad(fn_1)

In [ ]:
# reset .X (previously set to `None` in order to reduce file size)
adata_1.X = adata_1.layers["log_norm"]

In [ ]:
# check that column containing annotation labels is present
assert anno_col_1 in adata_1.obs.keys()

In [ ]:
adata_1.shape

In [ ]:
# (95772, 14796)

### Cell counts by annotation

In [ ]:
adata_1.obs[anno_col_1].value_counts()

In [ ]:
# CD4+ T      45070
# B           41056
# CD8+ T       7516
# Monocyte     1037
# NK            693
# pDC           400
# Name: anno_leiden_0.18, dtype: int64

### Dot plot (Figure S2B)

In [ ]:
genes_dict_1 = {
    "B": ["MS4A1", "CD19", "CD79A"],
    "T": ["CD3D", "CD3E", "CD3G", "IL7R"],
    "CD4+ T": ["CD4"],
    "CD8+ T": ["CD8A"],
    "NK": ["GZMB", "GNLY", "NKG7", "NCAM1"],
    "Monocyte": ["CD14", "LYZ", "CST3", "MS4A7"],
    "pDC": ["IL3RA", "CLEC4C"]
}

genes_lst_1 = [x for v in genes_dict_1.values() for x in v]

In [ ]:
# check that all genes for visualization are present in count matrix
assert all( [x in adata_1.var["gene_name"] for x in genes_lst_1] )

In [ ]:
anno_order_1 = ["B", "CD4+ T", "CD8+ T", "NK", "Monocyte", "pDC"]

In [ ]:
cur_fig = sc.pl.dotplot(adata_1, layer="log_norm", var_names=genes_dict_1, groupby=anno_col_1,
                        dendrogram=False, 
                        categories_order=anno_order_1, swap_axes=False,
                        cmap="Blues", return_fig=True, save=False)
cur_fig.savefig("fig_S2B.pdf", bbox_inches="tight")
plt.close()

### UMAPs (Figure S2A)

In [ ]:
# color palette
anno_palette_1 = {
    "B": "violet", 
    "CD4+ T": "dodgerblue",
    "CD8+ T": "orange",
    "NK": "purple", 
    "Monocyte": "seagreen", 
    "pDC": "darkgray"
}

In [ ]:
cur_fig = sc.pl.embedding(adata_1, basis=f"X_{eb}", color=anno_col_1, 
                          size=3, palette=anno_palette_1, 
                          legend_loc="right", legend_fontsize=0, legend_fontoutline=0,
                          frameon=True, ncols=1, title="",
                          return_fig=True, save=False)

cur_fig.savefig("fig_S2A.png", dpi=500, bbox_inches="tight")

plt.close(cur_fig)

In [ ]:
del adata_1

## B cells

Tip: When running the `B cells` section after running the `All cells` section, if memory limit is reached, skip running `All cells` and go straight to `B cells`. These two sections should be independent of each other.

### AnnData containing clustering results of B cells

In [ ]:
adata_2 = read_h5ad(fn_2)

In [ ]:
# reset .X (previously set to `None` in order to reduce file size)
adata_2.X = adata_2.layers["log_norm"]

In [ ]:
# check that column containing annotation labels is present
assert anno_col_2 in adata_2.obs.keys()

In [ ]:
# exclude non B-cell clusters
clusters_excl_2 = ["B & T", "Unassigned_1", "Unassigned_2"]
bool_incl_2 = ~adata_2.obs[anno_col_2].isin(clusters_excl_2)

# subset
adata_2 = adata_2[bool_incl_2, :]

# remove excluded categories
adata_2.obs[anno_col_2] = adata_2.obs[anno_col_2].cat.remove_unused_categories()

# re-run dendrogram
sc.tl.dendrogram(adata_2, groupby=anno_col_2, n_pcs=N_PC, use_rep="X_pca")

In [ ]:
adata_2.shape

In [ ]:
# (23128, 14796)

### Cell counts by annotation

In [ ]:
adata_2.obs[anno_col_2].value_counts()

In [ ]:
# MBC      12212
# Naive     8054
# GC        2018
# LNPC       844
# Name: anno_leiden_0.25, dtype: int64

### Dot plot (Figure S2D)

In [ ]:
genes_dict_2 = {
    "GC": ["BCL6", "RGS13", "MEF2B", "STMN1", "ELL3", "SERPINA9"],
    "PB/LNPC": ["XBP1", "IRF4", "SEC11C", "FKBP11", "JCHAIN", "PRDM1"],
    "Naive": ["TCL1A", "IL4R", "CCR7", "IGHM", "IGHD"],
    "MBC": ["TNFRSF13B", "CD27", "CD24"]
}

genes_lst_2 = [x for v in genes_dict_2.values() for x in v]

In [ ]:
# check that all genes for visualization are present in count matrix
assert all( [x in adata_2.var["gene_name"] for x in genes_lst_2] )

In [ ]:
anno_order_2 = ["Naive", "GC", "LNPC", "MBC"]

In [ ]:
cur_fig = sc.pl.dotplot(adata_2, layer="log_norm", var_names=genes_dict_2, groupby=anno_col_2,
                        dendrogram=False, 
                        categories_order=anno_order_2, swap_axes=False,
                        cmap="Blues", return_fig=True, save=False)
cur_fig.savefig("fig_S2D.pdf", bbox_inches="tight")
plt.close()

### UMAPs (Figure 2A, left; S2C)

In [ ]:
# color palette
anno_palette_2 = {
    "GC": "dodgerblue", 
    "LNPC": "forestgreen",
    "Naive": "orange",
    "MBC": "violet"
}

In [ ]:
cur_fig = sc.pl.embedding(adata_2, basis=f"X_{eb}", color=anno_col_2, 
                          size=3, palette=anno_palette_2, 
                          legend_loc="right", legend_fontsize=0, legend_fontoutline=0,
                          frameon=True, ncols=1, title="",
                          return_fig=True, save=False)

cur_fig.savefig("fig_2A_left_S2C.png", dpi=500, bbox_inches="tight")

plt.close(cur_fig)

### Export DataFrame

In [ ]:
# DataFrame containing UMAP coordinates and select metadata columns
eb_df_anno_2 = pd.concat(
    [ pd.DataFrame(adata_2.obsm[f"X_{eb}"], columns=[f"{eb}_x", f"{eb}_y"], index=adata_2.obs.index),
      adata_2.obs.loc[:, ["cell_id", anno_col_2, "donor", "sample"]] ], 
    axis=1)

In [ ]:
# used in R script for UMAP-based visualizations
eb_df_anno_2.to_csv(fn_save, sep="\t", header=True, index=True, compression="gzip")